- Links uteis:

1 - https://theairlab.org/alfa-dataset/ (contém informações sobre o status do vôo.)

# Explorando os dados


In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt


In [ ]:
def plot_flight_data_by_source(df, source_name, skip_noise=True):
    """
    df: DataFrame com o join completo
    source_name: String contida no nome das colunas (ex: 'battery' ou 'imu-data_raw')
    skip_noise: Se True, ignora colunas de covariância e metadados técnicos
    """
    # 1. Identificar colunas que pertencem a esta fonte
    # O padrão do join é: [NomeDoArquivo]_field.[Campo]
    cols = [c for c in df.columns if source_name in c and '_field.' in c]
    
    if skip_noise:
        noise_keywords = ['covariance', 'checksum', 'magic', 'seq', 'stamp', 'frame_id', 'sysid', 'compid', 'msgid']
        cols = [c for c in cols if not any(k in c.lower() for k in noise_keywords)]

    if not cols:
        print(f"Nenhuma coluna relevante encontrada para: {source_name}")
        return

    # 2. Configurar o layout dos gráficos
    n_cols = len(cols)
    fig, axes = plt.subplots(n_cols, 1, figsize=(15, 2.5 * n_cols), sharex=True)
    
    if n_cols == 1:
        axes = [axes]

    print(f"Plotando {n_cols} colunas de: {source_name}")
    
    # 3. Gerar os plots
    for i, col in enumerate(cols):
        clean_label = col.split('_field.')[-1] # Nome humano da variável
        axes[i].plot(df['%time'], df[col], label=clean_label, color='tab:blue')
        axes[i].set_ylabel(clean_label)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(loc='upper right')

    axes[-1].set_xlabel('Time (%time)')
    plt.suptitle(f"Análise de Sensores: {source_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    
    # Salvar ou mostrar
    #plt.savefig(f"plot_{source_name.replace('-', '_')}.png")
    plt.show()

In [ ]:
path = '../data/01_raw/processed/carbonZ_2018-07-18-15-53-31_2_engine_failure'
all_files = glob.glob(os.path.join(path, "*.csv"))

dfs = []

for filename in all_files:
    # Pega o nome do arquivo sem a extensão para usar como prefixo
    # Ex: 'engine_status' ou 'battery'
    topic_name = os.path.basename(filename).replace('.csv', '')
    
    temp_df = pd.read_csv(filename)
    
    # Identifica colunas numéricas
    cols = temp_df.select_dtypes(include=['number']).columns.tolist()
    if '%time' not in cols:
        temp_df['%time'] = pd.to_numeric(temp_df['%time'])
        cols.append('%time')
    
    temp_df = temp_df[cols].sort_values('%time')

    # RENOMEIA as colunas (exceto o tempo) para evitar colisão
    # field.data vira engine_status_field.data
    new_names = {c: f"{topic_name}_{c}" for c in temp_df.columns if c != '%time'}
    temp_df.rename(columns=new_names, inplace=True)
    
    dfs.append(temp_df)



In [ ]:
import pandas as pd

# 1. Encontrar o timestamp inicial absoluto de todos os arquivos antes de qualquer processamento
min_time = min([df['%time'].min() for df in dfs])

# 2. Normalizar o tempo e ordenar cada DataFrame individualmente
processed_dfs = []
for df in dfs:
    # Criar uma cópia para não alterar a lista original acidentalmente
    temp_df = df.copy()
    temp_df['%time'] = (temp_df['%time'] - min_time) / 1e9
    temp_df = temp_df.sort_values('%time')
    processed_dfs.append(temp_df)

# 3. Ordenar a lista pelo tamanho para garantir que o dataframe base seja o mais denso
processed_dfs.sort(key=len, reverse=True)

# 4. Inicializar o dataframe principal
merged_df = processed_dfs[0]

# 5. Realizar o join iterativo
for next_df in processed_dfs[1:]:
    # Identificar colunas que ainda não estão no merged_df
    cols_to_use = next_df.columns.difference(merged_df.columns).tolist()
    if '%time' not in cols_to_use:
        cols_to_use.append('%time')
    
    # merge_asof com direção 'backward' para garantir que NaNs apareçam 
    # se o sensor ainda não tiver começado a registrar naquele timestamp
    merged_df = pd.merge_asof(
        merged_df, 
        next_df[cols_to_use], 
        on='%time', 
        direction='backward'
    )

# 6. Preencher todos os valores ausentes com 0
# Isso afetará tanto o início (sensores atrasados) quanto falhas de log no meio
merged_df.fillna(0, inplace=True)

# 7. (Opcional) Converter colunas de status/inteiras que ficaram como float após o NaN
# merged_df = merged_df.infer_objects() 

print(f"Merge concluído!")
print(f"Shape final: {merged_df.shape}")
print(f"Eixo temporal: de {merged_df['%time'].min():.2f}s a {merged_df['%time'].max():.2f}s")

- Diferente de um join comum, o merge_asof com nearest é o mais indicado para sensores de drones por dois motivos:

- Latência de Sensores: O sensor de GPS pode registrar às 10.001s e o acelerômetro às 10.002s. O nearest garante que eles se alinhem no "momento" mais próximo.

- Preenchimento de NaNs: Se você usar left ou right puro, terá muitos buracos nos dados onde as frequências não batem.

In [ ]:
print(merged_df.columns.tolist())

In [ ]:
columns_field_data = [col for col in merged_df.columns if 'field.data' in col]

In [ ]:
columns_field_data

In [ ]:
merged_df[columns_field_data].head(20)

# Verificando colunas

In [ ]:
# Dicionário para organizar as colunas por arquivo
estrutura_colunas = {}

# Extrair as colunas associadas a cada fonte (arquivo CSV)
for col in merged_df.columns:
    if '_field.' in col:
        # Divide o nome: [Nome_do_Arquivo]_field.[Nome_da_Coluna]
        partes = col.split('_field.')
        fonte = partes[0].split('engine_failure-')[-1] # Pega só o nome do tópico/arquivo
        coluna_real = partes[1]
        
        if fonte not in estrutura_colunas:
            estrutura_colunas[fonte] = []
        estrutura_colunas[fonte].append(coluna_real)

# Imprimir de forma organizada para você copiar e me enviar
print("### ESTRUTURA DE COLUNAS POR ARQUIVO ###\n")
for arquivo in sorted(estrutura_colunas.keys()):
    print(f"Arquivo: {arquivo}")
    for c in estrutura_colunas[arquivo]:
        print(f"  - {c}")
    print("-" * 30)

# Filtrando apenas colunas e datasets que não são ruído

In [ ]:
# 1. Definir termos que indicam colunas de ruído/metadados para descarte
descarte_keywords = [
    'covariance', 'header.', 'checksum', 'magic', 'seq', 'stamp', 
    'frame_id', 'sysid', 'compid', 'msgid', 'payload', 'incompat_flags', 
    'compat_flags', 'len', 'framing_status', 'time_ref', 'serial_number'
]

# 2. Definir se preferimos o IMU processado ou o RAW
# (Escolhemos o processado por ser mais estável, mas você pode inverter)
fonte_imu_para_descartar = 'imu-data_raw' 

# 3. Lista para armazenar as colunas que vamos manter
colunas_uteis = ['%time'] # O tempo é essencial

for col in merged_df.columns:
    # Ignorar a coluna de tempo (já adicionada)
    if col == '%time':
        continue
        
    # Critério de descarte por palavra-chave
    if any(key in col.lower() for key in descarte_keywords):
        continue
        
    # Critério de descarte de redundância (IMU RAW)
    if fonte_imu_para_descartar in col:
        continue
        
    # Se passou pelos filtros, é uma coluna de relevância (Alta, Média ou Baixa)
    colunas_uteis.append(col)

# 4. Criar o DataFrame final filtrado
df_final = merged_df[colunas_uteis].copy()

# 5. Organizar as colunas alfabeticamente (exceto %time) para facilitar a inspeção
cols_sorted = ['%time'] + sorted([c for c in df_final.columns if c != '%time'])
df_final = df_final[cols_sorted]

print(f"Colunas originais: {merged_df.shape[1]}")
print(f"Colunas após limpeza: {df_final.shape[1]}")
print("\nExemplos de colunas mantidas:")
print(df_final.columns[:10].tolist())

# 1 - Fontes a verificar

In [ ]:
# Descobrir todas as fontes (arquivos originais) presentes no join
fontes = set([c.split('_field.')[0] for c in df_final.columns if '_field.' in c])
print("Fontes detectadas no merge:")
for f in sorted(fontes):
    # Exibe apenas a parte final do nome para facilitar a leitura
    print(f" - {f.split('engine_failure-')[-1]}")



In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'diagnostics', 
    'failure_status-engines', 
    'mavctrl-path_dev', 
    'mavlink-from'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df_final.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df_final[colunas_selecionadas].columns.tolist())



In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df_final, p)

In [ ]:
# 1. Identificar as colunas que queremos remover
colunas_para_remover = [col for col in df_final.columns 
                        if 'diagnostics' in col or 'mavlink-from' in col]

# 2. Remover do DataFrame
df1 = df_final.drop(columns=colunas_para_remover)

print(f"Colunas removidas: {len(colunas_para_remover)}")
print(f"Colunas restantes no dataset: {df1.shape[1]}")

# 3. Verificar o que sobrou (opcional)
# print(df_final.columns.tolist())

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'diagnostics', 
    'failure_status-engines', 
    'mavctrl-path_dev', 
    'mavlink-from'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df1.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df1[colunas_selecionadas].columns.tolist())

# Explicação de algumas colunas

1. failure_status-engines_field.data

    O que é: O Gabarito (Ground Truth).

    Explicação: É um sinal booleano (0 ou 1). No instante em que este valor passa de 0 para 1, a falha de motor foi injetada no sistema.

    Uso na IA: É a variável que você quer prever. O objetivo do seu modelo é identificar mudanças nos outros sinais (como os desvios abaixo) que ocorram simultaneamente ou logo após esse 1 aparecer.

2. mavctrl-path_dev_field.x (Desvio Longitudinal)

    O que é: Erro de posição no eixo de movimento (frente/trás).

    Explicação: Representa a distância entre onde o drone deveria estar na linha do tempo da missão e onde ele realmente está ao longo da trajetória.

    Comportamento na Falha: Quando o motor para, o drone perde velocidade. Ele começa a ficar "atrasado" em relação ao ponto ideal da missão. Você verá o valor de x aumentar (ou ficar mais negativo, dependendo da convenção), indicando que o drone não consegue mais acompanhar o ritmo planejado.

3. mavctrl-path_dev_field.y (Desvio Lateral)

    O que é: Erro de posição lateral (esquerda/direita).

    Explicação: Mede o quanto o drone saiu "para o lado" da linha reta da missão (o Cross-track error).

    Comportamento na Falha: Em aviões (asa fixa), a perda de motor pode gerar um torque assimétrico ou, mais comum, a perda de autoridade de controle das superfícies (leme/aileron) devido à baixa velocidade. Se houver vento lateral, o drone será "empurrado" para fora da rota, e o valor de y subirá drasticamente porque o drone não tem mais propulsão para corrigir o curso.

4. mavctrl-path_dev_field.z (Desvio Vertical / Altitude)

    O que é: Erro de altitude.

    Explicação: A diferença entre a altitude desejada e a altitude real.

    Comportamento na Falha: Este é o indicador mais dramático em aeronaves de asa fixa. Sem motor, não há sustentação suficiente para manter o voo nivelado. O drone começa a perder altura.

    O Gráfico: Se a missão pede 50m de altura e o drone cai para 40m, o path_dev.z mostrará um erro de 10m. É a confirmação física de que o drone está em um mergulho não planejado.

# 2 - Fontes a verificar

 - mavros-battery
 - mavros-global_position-compass_hdg
 - mavros-global_position-global
 - mavros-global_position-local
 - mavros-global_position-raw-fix
 - mavros-global_position-raw-gps_vel
 - mavros-global_position-rel_alt
 - mavros-imu-atm_pressure
 - mavros-imu-data
 - mavros-imu-mag
 - mavros-imu-temperature
 - mavros-local_position-odom
 - mavros-local_position-pose
 - mavros-local_position-velocity
 - mavros-nav_info-airspeed
 - mavros-nav_info-errors
 - mavros-nav_info-pitch
 - mavros-nav_info-roll
 - mavros-nav_info-velocity
 - mavros-nav_info-yaw
 - mavros-rc-in
 - mavros-rc-out
 - mavros-setpoint_raw-local
 - mavros-setpoint_raw-target_global
 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'mavros-battery', 
    'mavros-global_position-compass_hdg', 
    'mavros-global_position-global', 
    'mavros-global_position-local'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df1.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df1[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. mavros-battery (Estado Elétrico)

Como você já observou que a bateria é estável no ALFA Dataset, a maioria dessas colunas será constante, mas duas são vitais para confirmar que não houve um "apagão":

- current: Corrente elétrica. Mesmo estável, ela deve cair levemente se o motor para de consumir energia.

- voltage: Tensão. Se o motor trava e tenta girar (curto), a voltagem cai (sag). Se o circuito abre, a voltagem sobe levemente.

- percentage: Útil apenas para saber o estado geral de carga.

- Descarte: design_capacity, location, serial_number, technology. São metadados estáticos (não mudam nunca).

2. mavros-global_position-global & compass_hdg

Aqui temos os dados do GPS e da bússola magnética.

- compass_hdg_field.data: O Heading (direção) da bússola. Se o motor falha e o avião começa a girar (guinada), este valor mudará de forma não planejada.

- altitude: Altitude absoluta (MSL). Crucial para ver a queda do avião.

- latitude / longitude: Úteis para plotar o mapa do voo, mas péssimas para modelos de IA (números muito grandes e mudanças pequenas dificultam o aprendizado).

- status.status: Indica se o GPS está com sinal (fix). Se o motor causar interferência eletromagnética ao falhar, o status pode oscilar.

3. mavros-global_position-local (Cinemática do Voo)

Este é o arquivo mais rico em física depois do IMU. Ele transforma os dados globais em um plano local (X, Y, Z), o que é muito melhor para a IA processar.

- pose.pose.position.z: É a altitude local. Muito mais fácil de usar que a altitude global para detectar quedas.

- twist.twist.linear.x, y, z: Velocidades lineares.

    - O linear.x é a velocidade de avanço. Se o motor para, ela cai.

    - O linear.z é a velocidade vertical. Se o motor para, ela fica negativa (descida).

- twist.twist.angular.x, y, z: Velocidades angulares. Medem o "balanço" do drone. Se o avião começar a "capotar" ou girar após a falha, esses valores explodem.

- pose.pose.orientation.w, x, y, z: A atitude do drone (Quatérnios). Representam a inclinação.

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df1, p)

In [ ]:
# 1. Lista de termos que identificam colunas irrelevantes (metadados e status estáticos)
termos_inuteis = [
    'design_capacity', 'location', 'present', 'serial_number', 
    'technology', 'power_supply_health', 'power_supply_technology',
    'status.service', 'capacity', 'charge'
]

# 2. Identificar todas as colunas no merged_df que contenham algum desses termos
colunas_para_deletar = [col for col in df1.columns 
                        if any(termo in col.lower() for termo in termos_inuteis)]

# 3. Remover as colunas do DataFrame
df2 = df1.drop(columns=colunas_para_deletar)

# 4. Remover também Latitude e Longitude se você for usar apenas a posição LOCAL
# GPS global costuma ter números muito grandes que atrapalham redes neurais
colunas_gps_global = [col for col in df2.columns if 'latitude' in col or 'longitude' in col]
df2 = df2.drop(columns=colunas_gps_global)

print(f"Total de colunas deletadas: {len(colunas_para_deletar) + len(colunas_gps_global)}")
print(f"Colunas restantes para análise: {df2.shape[1]}")


In [ ]:
# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df2.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df1[colunas_selecionadas].columns.tolist())

# 3 - Fontes a verificar

 - mavros-global_position-raw-fix
 - mavros-global_position-raw-gps_vel
 - mavros-global_position-rel_alt
 - mavros-imu-atm_pressure
 - mavros-imu-data
 - mavros-imu-mag
 - mavros-imu-temperature
 - mavros-local_position-odom
 - mavros-local_position-pose
 - mavros-local_position-velocity
 - mavros-nav_info-airspeed
 - mavros-nav_info-errors
 - mavros-nav_info-pitch
 - mavros-nav_info-roll
 - mavros-nav_info-velocity
 - mavros-nav_info-yaw
 - mavros-rc-in
 - mavros-rc-out
 - mavros-setpoint_raw-local
 - mavros-setpoint_raw-target_global
 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'mavros-global_position-raw-fix', 
    'mavros-global_position-raw-gps_vel', 
    'mavros-global_position-rel_alt', 
    'mavros-imu-atm_pressure'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df2.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df1[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. Dados de Altitude e Pressão (Verticalidade)

- raw-fix_field.altitude: É a altitude bruta medida pelo GPS (em relação ao nível do mar).

    - Na falha: Esse número deve diminuir conforme o avião perde sustentação.

- rel_alt_field.data: Altitude Relativa. É a altura do drone em relação ao ponto de decolagem.

    - Relevância: Alta. Para IAs, essa coluna é melhor que a altitude absoluta porque começa em 0 ou perto disso, facilitando a detecção de quedas.

- atm_pressure_field.fluid_pressure: Pressão atmosférica medida pelo barômetro.

    - Física: Conforme o avião cai (perde altitude), a pressão atmosférica aumenta.

    - Importância: É um sensor independente do GPS. Se o GPS falhar, a pressão te diz se o avião está caindo.

2. Velocidade e Movimento GPS (Cinemática Bruta)

Essas colunas vêm do arquivo raw-gps_vel e mostram o deslocamento medido diretamente pelos satélites:

- twist.linear.x, y, z: Velocidades lineares no plano terra.

    - linear.x: Velocidade para o Norte/Frente.

    - linear.z: Velocidade vertical (Subida/Descida). Se o motor para, este valor fica negativo (indicando descida).

- twist.angular.x, y, z: Velocidades angulares (Taxa de rotação).

    - Nota: No GPS, esses valores costumam ser menos precisos que os da IMU (Giroscópio), mas ajudam a entender se o avião entrou em um "parafuso" ou está girando sem controle após a perda de potência.

3. Status e Ruído

- raw-fix_field.status.status: Indica a qualidade do sinal GPS (ex: 0 = No Fix, 2 = Fix, 3 = DGPS).

    - Uso: Serve apenas para validar se os dados de altitude acima são confiáveis.

- atm_pressure_field.variance: O "ruído" ou erro estimado do sensor de pressão.

    - Relevância: Baixa. Pode ser descartada, pois não descreve o voo, apenas a precisão do sensor.

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df2, p)

In [ ]:
# 1. Lista de termos que identificam colunas irrelevantes (metadados e status estáticos)
termos_inuteis = [
    'status.status', 'variance', 'twist.linear.z', 'twist.angular.x', 
    'twist.angular.y', 'twist.angular.z', 'fluid_pressure'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df2.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)


# 2. Identificar todas as colunas no merged_df que contenham algum desses termos
colunas_para_deletar = [col for col in df2[colunas_selecionadas].columns 
                        if any(termo in col.lower() for termo in termos_inuteis)]

# 3. Remover as colunas do DataFrame
df3 = df2.drop(columns=colunas_para_deletar)


print(f"Total de colunas deletadas: {len(colunas_para_deletar)}")
print(f"Colunas restantes para análise: {df3.shape[1]}")


In [ ]:
# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df3.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df1[colunas_selecionadas].columns.tolist())

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df3, p)

# 4 - Fontes a verificar

 - mavros-imu-data
 - mavros-imu-mag
 - mavros-imu-temperature
 - mavros-local_position-odom
 - mavros-local_position-pose
 - mavros-local_position-velocity
 - mavros-nav_info-airspeed
 - mavros-nav_info-errors
 - mavros-nav_info-pitch
 - mavros-nav_info-roll
 - mavros-nav_info-velocity
 - mavros-nav_info-yaw
 - mavros-rc-in
 - mavros-rc-out
 - mavros-setpoint_raw-local
 - mavros-setpoint_raw-target_global
 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'mavros-imu-data', 
    'mavros-imu-mag', 
    'mavros-imu-temperature', 
    'mavros-local_position-odom'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df3.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df3[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. mavros-imu-data (O Labirinto Interno)

A IMU (Unidade de Medida Inercial) é o sensor mais rápido. Ela sente a falha antes mesmo do GPS perceber que o drone mudou de lugar.

- angular_velocity.x, y, z: Medem a velocidade de rotação (rad/s).

    - Importância: Se o motor falha, o avião pode dar uma guinada ou começar a balançar. Picos súbitos aqui indicam perda de estabilidade.

- linear_acceleration.x, y, z: Medem forças G (m/s²).

    - X (Frente): Sobe no mergulho.

    - Z (Vertical): Em voo reto e nivelado, vale aproximadamente 9.81m/s2. Se o valor cair drasticamente, o drone está em "queda livre".

- orientation.w, x, y, z: Representam a inclinação no espaço usando Quatérnios.

    - Dica: É difícil para humanos lerem quatérnios, mas para a IA eles são ótimos porque não sofrem de travamento de eixos (Gimbal Lock).

2. mavros-imu-mag e imu-temperature

- magnetic_field.x, y, z: O magnetômetro (bússola digital).

    - Na Falha: Motores elétricos geram campos magnéticos. Se o motor para de girar bruscamente, você pode ver um "degrau" ou mudança no ruído magnético nessas colunas.

- temperature: Temperatura interna do sensor.

    - Relevância: Baixa. Geralmente usada para compensar erros térmicos dos sensores, não para detectar falha de motor (a menos que o motor pegue fogo, mas o sensor está longe dele).

3. mavros-local_position-odom (Odometria)

Este arquivo combina GPS e IMU para criar uma estimativa suave de onde o drone está. É a fonte de dados mais confiável para navegação.

- pose.pose.position.x, y, z: Posição relativa ao ponto de decolagem (em metros).

    - position.z: É a sua altitude local. Se o motor falha, este valor diminui constantemente.

- pose.pose.orientation.w, x, y, z: Semelhante à IMU, mas "filtrada". Mostra a atitude (inclinação) do drone de forma mais estável.

- twist.twist.linear.x, y: Velocidade de deslocamento no plano horizontal.

    - Se o motor falha, a velocidade em X (avanço) começa a cair devido ao arrasto do ar.

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df3, p)

In [ ]:
# 1. Lista de termos que identificam colunas irrelevantes (metadados e status estáticos)
termos_inuteis = [
    'temperature', 'variance'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df3.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)


# 2. Identificar todas as colunas no merged_df que contenham algum desses termos
colunas_para_deletar = [col for col in df3[colunas_selecionadas].columns 
                        if any(termo in col.lower() for termo in termos_inuteis)]

# 3. Remover as colunas do DataFrame
df4 = df3.drop(columns=colunas_para_deletar)


print(f"Total de colunas deletadas: {len(colunas_para_deletar)}")
print(f"Colunas restantes para análise: {df4.shape[1]}")


In [ ]:
# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df4.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df1[colunas_selecionadas].columns.tolist())

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df4, p)

# 5 - Fontes a verificar

 - mavros-local_position-pose
 - mavros-local_position-velocity
 - mavros-nav_info-airspeed
 - mavros-nav_info-errors
 - mavros-nav_info-pitch
 - mavros-nav_info-roll
 - mavros-nav_info-velocity
 - mavros-nav_info-yaw
 - mavros-rc-in
 - mavros-rc-out
 - mavros-setpoint_raw-local
 - mavros-setpoint_raw-target_global
 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'mavros-local_position-pose', 
    'mavros-local_position-velocity', 
    'mavros-nav_info', 
    'mavros-nav_info-errors'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df4.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df4[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. nav_info-airspeed e nav_info-errors (O Coração da Falha)

Estas são as colunas mais importantes para detectar a perda de motor em aviões.

- airspeed_field.measured: A velocidade real do ar. Cai constantemente após a falha.

- airspeed_field.commanded: A velocidade que o drone deveria ter. Geralmente permanece alta, pois o controlador tenta manter o voo.

- errors_field.aspd_error: A diferença entre a desejada e a real. Explode positivamente na falha.

- errors_field.alt_error: Erro de altitude. Quando o avião começa a mergulhar sem motor, este erro aumenta drasticamente.

- errors_field.xtrack_error: Erro lateral. Indica se o avião está sendo empurrado para fora da linha da missão.

2. nav_info-pitch / roll / yaw / velocity (Desejado vs. Medido)

Aqui se tem o par commanded (o que o software quer) e measured (o que os sensores sentem).

- pitch_field.commanded: Na falha, o piloto automático pode comandar um pitch positivo (nariz para cima) para tentar manter a altitude, enquanto o measured pode mostrar o nariz caindo (negativo) por falta de velocidade.

- velocity_field.des_x, y, z: Velocidades desejadas nos eixos locais.

- velocity_field.meas_x, y, z: Velocidades reais medidas. A discrepância entre des_z (desejado subir/manter) e meas_z (real descendo) é um indicador crítico.

3. local_position-pose e local_position-velocity

Estes são dados de navegação local (coordenadas cartesianas em metros).

- pose.position.x, y, z: Posição relativa ao ponto de origem. O z é a altitude local, essencial para monitorar a queda.

- pose.orientation.w, x, y, z: A atitude do drone (como ele está inclinado) em Quatérnios.

- velocity_field.twist.linear.x, y, z: Velocidade de deslocamento em metros por segundo.

    - linear.z: Velocidade vertical. Valores negativos altos indicam uma queda rápida.

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df4, p)

In [ ]:
# 1. Lista de termos que identificam colunas irrelevantes (metadados e status estáticos)
termos_inuteis = [
    'coordinate_frame'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df4.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)


# 2. Identificar todas as colunas no merged_df que contenham algum desses termos
colunas_para_deletar = [col for col in df4[colunas_selecionadas].columns 
                        if any(termo in col.lower() for termo in termos_inuteis)]

# 3. Remover as colunas do DataFrame
df5 = df4.drop(columns=colunas_para_deletar)


print(f"Total de colunas deletadas: {len(colunas_para_deletar)}")
print(f"Colunas restantes para análise: {df5.shape[1]}")


In [ ]:
# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df5.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df5[colunas_selecionadas].columns.tolist())

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df5, p)

# 6 - Fontes a verificar

 - mavros-nav_info-pitch
 - mavros-nav_info-roll
 - mavros-nav_info-velocity
 - mavros-nav_info-yaw
 - mavros-rc-in
 - mavros-rc-out
 - mavros-setpoint_raw-local
 - mavros-setpoint_raw-target_global
 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
 'mavros-nav_info-pitch',
 'mavros-nav_info-roll',
 'mavros-nav_info-velocity',
 'mavros-nav_info-yaw'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df5.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df5[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. Pitch, Roll e Yaw (Atitude do Nariz e Asas)

Essas colunas descrevem a orientação angular do avião em graus ou radianos.

- pitch_field (Arfagem): Movimento do nariz para cima ou para baixo.

    - commanded: O que o piloto automático quer. Na falha, ele pode comandar "nariz para cima" para evitar perder altitude.

    - measured: O que realmente acontece. Geralmente o nariz cai devido à perda de velocidade (estol).

- roll_field (Rolagem): Inclinação das asas para esquerda ou direita.

    - Na Falha: Se houver vento ou se o motor gerar um tranco ao parar, o measured mostrará oscilações enquanto o commanded tenta manter as asas niveladas.

- yaw_field (Guinada): Direção do nariz (bússola).

    - Na Falha: O avião pode "caranguejar" ou girar lateralmente se perder a propulsão simétrica.

2. Velocity X, Y, Z (Vetores de Velocidade Local)

Aqui medimos o deslocamento linear em metros por segundo (m/s).

- velocity_field.des_x, y, z: É o vetor de velocidade que o controlador planejou para aquele exato momento da missão.

- velocity_field.meas_x, y, z: É a velocidade real calculada pela fusão de sensores (GPS + IMU).

    - Eixo X (Frente/Trás): O meas_x cai drasticamente após a falha.

    - Eixo Z (Cima/Baixo): Este é o mais revelador. O des_z pode ser zero (manter altura), mas o meas_z se torna negativo e grande (descida rápida).

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df5, p)

# 7 - Fontes a verificar

 - mavros-rc-in
 - mavros-rc-out
 - mavros-setpoint_raw-local
 - mavros-setpoint_raw-target_global
 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'mavros-rc-in', 
    'mavros-rc-out', 
    'mavros-setpoint_raw-local', 
    'mavros-setpoint_raw-target_global'
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df5.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df5[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. mavros-rc-in (Comandos do Piloto/Rádio)

Estes são os sinais que entram no drone, vindos do controle remoto (ou simulados).

- channels0 a channels15: Representam as alavancas e botões do rádio.

    - Canal 2 (geralmente Throttle): Se o piloto estiver tentando acelerar manualmente, este valor muda.

    - Canal 0, 1, 3: Geralmente Roll, Pitch e Yaw.

- rssi: Indica a força do sinal de rádio.

    - Relevância: Se o rssi cair para zero, a falha pode ser de comunicação (perda de link) e não mecânica.

2. mavros-rc-out (Comandos para os Motores/Servos)

Este é o dado mais importante deste grupo. Ele mostra o que o piloto automático está enviando "de fato" para o hardware.

- channels: Valores em PWM (geralmente entre 1000 e 2000).

    - Canal 2 (Esc/Motor): Se o motor falhou mas o valor aqui for 2000 (máximo), confirma que o drone está tentando desesperadamente girar o motor, mas não há resposta física.

    - Outros Canais: Representam os movimentos dos servos das asas (ailerons, profundor).

3. setpoint_raw (As Metas do Computador)

Diferente dos dados de navegação que vimos antes (que medem o que está acontecendo), os Setpoints são o "plano de futuro" imediato.

- local_field e target_global_field:

    - position.x, y, z: A coordenada exata onde o drone quer estar agora.

    - velocity.x, y, z: A velocidade que o drone quer atingir.

    - acceleration_or_force: O "empurrão" que o software planeja dar para chegar na meta.

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df5, p)

In [ ]:
# 1. Lista de termos específicos para exclusão
# Dica: use o nome do arquivo + o campo para ser cirúrgico na deleção
termos_inuteis = [
    # Removendo especificamente o yaw_rate do SETPOINT_RAW-LOCAL
    'setpoint_raw-local_field.yaw_rate', 
    'setpoint_raw-local_field.yaw', 
    'setpoint_raw-local_field.acceleration_or_force.x', 
    'setpoint_raw-local_field.acceleration_or_force.y', 
    'setpoint_raw-local_field.acceleration_or_force.z', 
    'velocity.z',
  
    
    # Outros termos genéricos que você quer remover de todos
    'type_mask',
    'coordinate_frame',
    'altitude',
    'acceleration_or_force',
    'header.frame_id',
    'header.seq',
    'header.stamp',
    'position.x',
    'position.y',
    'position.z',
    
    # Canais de rádio (RC-IN) que você listou com hifens antes
    'channels0', 'channels1', 'channels2', 'channels3', 'channels4',
    'channels5', 'channels6', 'channels7', 'channels8', 'channels9',
    'channels10', 'channels11', 'channels12', 'channels13', 'channels14', 'channels15',
    'rssi'
]

# Criar uma lista com todas as colunas que pertencem aos seus arquivos alvo
colunas_selecionadas = []
for col in df5.columns:
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# 2. Identificar as colunas para deletar
colunas_para_deletar = [
    col for col in df5[colunas_selecionadas].columns 
    if any(termo in col.lower() for termo in termos_inuteis)
]

# 3. Remover as colunas
df6 = df5.drop(columns=colunas_para_deletar)

print(f"Total de colunas deletadas: {len(colunas_para_deletar)}")
print(f"Colunas restantes: {df6.shape[1]}")

# Verificação: o yaw_rate do target_global ainda existe?
yaw_restante = [c for c in df6.columns if 'yaw_rate' in c]
print(f"\nColunas de 'yaw_rate' que sobraram: {yaw_restante}")


In [ ]:
# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df6.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df6[colunas_selecionadas].columns.tolist())

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df6, p)

# 8 - Fontes a verificar

 - mavros-state
 - mavros-vfr_hud
 - mavros-wind_estimation

In [ ]:
# Lista dos nomes curtos dos arquivos que você quer filtrar
arquivos_alvo = [
    'mavros-state', 
    'mavros-vfr_hud', 
    'mavros-wind_estimation', 
]

# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df6.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df6[colunas_selecionadas].columns.tolist())



## Explicação das colunas

1. mavros-state (Estado Lógico)

Essas colunas indicam o "pensamento" do software de controle.

- armed: Booleano. Indica se os motores estão autorizados a girar. Se mudar para False em pleno voo, você teve um crash de software ou desarmamento de emergência.

- connected: Indica se o MAVROS está recebendo dados do controlador de voo.

- guided: Indica se o drone está seguindo uma missão autônoma (Waypoints).

- system_status: Um código numérico da saúde do sistema MAVLink (ex: 3 = Standby, 4 = Active).

2. mavros-vfr_hud (Instrumentos Digitais de Voo)

O VFR HUD (Visual Flight Rules Head-Up Display) é o resumo dos principais sensores. É a coluna vertebral para entender o comportamento do avião.

- airspeed: Velocidade em relação ao ar. Crucial: cai quando o motor para.

- groundspeed: Velocidade em relação ao solo.

- throttle: O acelerador (0 a 100%). Destaque: No ALFA Dataset, você verá o throttle subir para 100% enquanto o airspeed cai, o que confirma a falha mecânica.

- climb: Taxa de subida/descida em m/s. Valores negativos indicam que o avião está perdendo altitude.

- altitude: Altitude atual.

- heading: A direção (bússola) para onde o nariz aponta.

3. mavros-wind_estimation (Estimativa de Vento)

O controlador de voo usa a diferença entre a velocidade do solo (GPS) e a velocidade do ar (Pitot) para calcular o vento externo.

- twist.linear.x, y, z: Velocidade estimada do vento nos eixos.

- twist.angular.x, y, z: Turbulência ou variação na direção do vento.

- Importância: Se o linear.x (vento de frente) mudar bruscamente sem motivo meteorológico logo após a falha, pode ser um erro de cálculo do sensor causado pela queda de velocidade do avião.

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df6, p)

In [ ]:
# 1. Lista de termos específicos para exclusão
# Dica: use o nome do arquivo + o campo para ser cirúrgico na deleção
termos_inuteis = [

    'twist.angular.x',
    'twist.angular.y',
    'twist.angular.z',
    'armed',
    'connected',
    'guided',
    'system_status'
]

# Criar uma lista com todas as colunas que pertencem aos seus arquivos alvo
colunas_selecionadas = []
for col in df6.columns:
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# 2. Identificar as colunas para deletar
colunas_para_deletar = [
    col for col in df6[colunas_selecionadas].columns 
    if any(termo in col.lower() for termo in termos_inuteis)
]

# 3. Remover as colunas
df7 = df6.drop(columns=colunas_para_deletar)

print(f"Total de colunas deletadas: {len(colunas_para_deletar)}")
print(f"Colunas restantes: {df7.shape[1]}")



In [ ]:
# Criar uma lista com todas as colunas que pertencem a esses arquivos
colunas_selecionadas = []
for col in df7.columns:
    # Verifica se o nome de algum dos arquivos alvo está presente no nome da coluna
    if any(arquivo in col for arquivo in arquivos_alvo):
        colunas_selecionadas.append(col)

# Exibir as colunas encontradas
print(f"Foram encontradas {len(colunas_selecionadas)} colunas.")
print(df7[colunas_selecionadas].columns.tolist())

In [ ]:
for p in arquivos_alvo:
    plot_flight_data_by_source(df7, p)

In [ ]:
output_dir = '../data/02_intermediate'

# Salvar o dataframe completo (Master)
file_path = os.path.join(output_dir, 'carbonZ_2018-07-18-15-53-31_1_engine_failure_pre_processed.csv')
df7.to_csv(file_path, index=False)

print(f"Dataset mestre salvo com sucesso em: {file_path}")

In [ ]:
df7.info()